## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [1]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama 
from langchain_classic.memory import ChatMessageHistory
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain, LLMChain, SequentialChain
from langchain_classic.chains.summarize import load_summarize_chain

KeyboardInterrupt: 

### 1 & 2. Model and Chat Model 

In [43]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [41]:
def get_chat_llm (params=None):
    
    llm = ChatGoogleGenerativeAI(
        model = "gemini-2.5-flash",
        temperature=0.3,
        api_key=Model_API
    )
    
   # llm = Ollama(model="llama3.1")
    
    return llm

In [44]:
llm_model = get_chat_llm()

### 3. Chat Message 

In [7]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [8]:
print(message)

Based on your preference for a sport sedan, I would recommend the BMW M3 as it offers excellent performance, handling, and style with its powerful engine and agile suspension.


### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [7]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [8]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.3, client=<google.genai.client.Client object at 0x000001F46FED8150>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

#### Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [10]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [11]:
response = llm_model.invoke(formatted_prompt)
response

AIMessage(content="The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).\n\n---\n\nThe **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider':

In [12]:
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).

---

The **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.


In [13]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**.

---

The **Toyota Supra** is an iconic Japanese sports car known for its performance, rear-wheel drive layout, and powerful inline-six engines. The **fourth-generation (Mk4)**, produced from 1993-2002, is particularly legendary, largely due to its highly tunable 2JZ-GTE engine and appearances in popular culture like the *Fast & Furious* film series. After a long hiatus, the Supra returned in 2019 (the **fifth-generation or A90/A91**) as a collaboration with BMW, maintaining its focus on performance and driver engagement.


### 5. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [14]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [15]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [16]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:


### 6. Output Parsers

Output parsers are used to convert the raw responses generated by an LLM into a more structured and useful format. They are especially helpful when working with structured data or when standardizing outputs from different language and chat models.

LangChain provides a variety of output parsers to handle different formatting needs. In this lab, you will work with the following two parsers:

* JSON Parser: Produces output in JSON format according to a specified schema. It can also utilize a Pydantic model to generate structured JSON data, making it one of the most dependable options for obtaining structured output without relying on function calling.

* CSV Parser: Formats the model's output as a list of comma-separated values (CSV), which is useful for tabular data representation.

In [17]:
class Car(BaseModel):
    query:str = Field(description="question to setup a car")
    response:str = Field(description="answer to resolve the car")

In [18]:
car_query = "tell me the information about Lambogini"

output_parser = JsonOutputParser(pydantic_object=Car)
format_instruction = output_parser.get_format_instructions()

prompt  = PromptTemplate(
    template= "Answer the user query.\n {format_instruction}\n{query}",
    input_variables=['query'],
    partial_variables= {"format_instruction": format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":car_query })
print(response)


{'query': 'tell me the information about Lambogini', 'response': "Lamborghini (officially Automobili Lamborghini S.p.A.) is an Italian brand and manufacturer of luxury sports cars and SUVs based in Sant'Agata Bolognese. The company is owned by the Volkswagen Group through its subsidiary Audi. Founded by Ferruccio Lamborghini in 1963, the company was created with the aim of competing with established marques like Ferrari. Lamborghini is renowned for its high-performance vehicles, often characterized by their distinctive, aggressive styling and powerful engines, typically V10 or V12. Iconic models include the Miura, Countach, Diablo, Murciélago, Gallardo, Aventador, Huracán, and the Urus SUV."}


In [19]:
user_query = "Give me details about DPO fine-tune"

output_parser = CommaSeparatedListOutputParser()
format_instruction = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query\n{format_instruction}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instruction":format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":user_query})
print(response)

['Direct Preference Optimization', 'Aligns LLMs with human preferences', 'Does not require a separate reward model', 'Uses preference data (chosen vs. rejected responses)', 'Simpler and more stable than PPO-based RLHF', 'Optimizes a direct analytical loss function', 'Based on a supervised fine-tuned (SFT) model', 'Computationally efficient', 'Achieves comparable or superior performance to RLHF', 'Avoids complexities of reinforcement learning']


## **LangChain Components for RAG Application**

### 7. Documents

#### Document Object

The document object contains two main components:

1. page-content: this represents the content of the whole document.
2. metadata: this represents the all the attributes that paticularlly unique for the each document. this availables at Dict types

examples:
document_id, timestamp, document_title, author, etc

In [20]:
document = Document(
    page_content="This is a example for document attribute which is page_content",
    metadata={
        "document_id": 1294857,
        "document_source": "about firewall",
        "created_time": 21344656
    }
)

document.page_content

'This is a example for document attribute which is page_content'

#### Document Loaders

Document loaders in LangChain are used to import data from multiple sources. For example, you can load a PDF file and make it readable for an LLM using LangChain.

LangChain provides more than 100 different document loaders, along with integrations with major platforms like AirByte and Unstructured. These integrations allow you to load various types of content such as HTML, PDFs, and source code from different locations, including private S3 buckets and public websites.

A full list of supported document types can be found here: https://python.langchain.com/v0.1/docs/integrations/document_loaders/

In [12]:
file_path = "https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf"
loader = PyPDFLoader(file_path)

In [13]:
loader

In [14]:
document = loader.load()
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tool

#### URL and Website Loaders

In [9]:
web_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()

In [10]:
web_data[0].page_content[:1000]

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain provides create_agent: a minimal, highly configurable age

#### Text-Splitter

After loading documents, you often need to transform them so they work better for your specific use case.

A common step is breaking a long document into smaller chunks that can fit within a model’s context window. LangChain provides several built-in document transformers that help with splitting, merging, filtering, and other ways of processing documents.

In general, text splitters work in the following way:

1. The text is first divided into small, meaningful units, such as sentences.
2. These small parts are then gradually combined into larger chunks until a defined size limit is reached (based on a chosen measurement method).
3. Once a chunk reaches the limit, it is stored as a separate segment, and a new chunk begins. Often, a small overlap is kept between chunks to preserve context across sections.

You can view the different types of text splitters supported by LangChain here:
[https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/)

As an example, we will use the simple `CharacterTextSplitter` to divide the LangChain paper you previously loaded. This basic splitter works by breaking text using characters (defaulting to `"\n\n"`) and measuring chunk size based on the number of characters.


In [15]:
text_splitter = CharacterTextSplitter(chunk_size = 700, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))

60


In [16]:
print(chunks[8].page_content)

deployment of LangChain applications. Finally, section 5 addresses the limita-
tions and criticisms of LangChain, particularly focusing on the complexities and
security concerns associated with its modular design and external integrations.
1 Architecture
LangChain is built with a modular architecture, designed to simplify the life-
cycle of applications powered by large language models (LLMs), from initial
development through to deployment and monitoring [3]. This modularity al-
lows developers to configure, extend, and deploy applications tailored to specific


#### Embedding Models

Embedding models are designed to work directly with text embeddings.

They convert a piece of text into a numerical vector representation. This is useful because it lets you represent text in a vector space, where similar meanings are positioned closer together. As a result, you can perform tasks like semantic search by finding text segments that are closest to each other in that vector space.


In [5]:
embeddings = HuggingFaceEmbeddings(
    model="BAAI/bge-m3",
    
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [6]:
text = "Hi im seran"
embeds = embeddings.embed_query("Hi im seran")
embeds

[-0.03832868114113808,
 0.018502652645111084,
 -0.04393254965543747,
 -0.0003660297370515764,
 -0.011765962466597557,
 -0.01035870797932148,
 -0.01746307872235775,
 0.005218490958213806,
 0.003107429714873433,
 0.015090255066752434,
 0.01508807111531496,
 0.009850089438259602,
 0.000960557081270963,
 -0.001512787421233952,
 -0.023006483912467957,
 -0.009658196941018105,
 0.03315352648496628,
 -0.02878696098923683,
 0.04440535604953766,
 0.022929757833480835,
 -0.0315418504178524,
 0.00619888911023736,
 -0.008088978007435799,
 0.004414239898324013,
 0.03774955868721008,
 0.052773598581552505,
 -0.015159388072788715,
 0.04066237062215805,
 -0.024652661755681038,
 -0.009521335363388062,
 -0.00772809935733676,
 0.02571774832904339,
 -0.0014926429139450192,
 -0.0699140727519989,
 -0.0077828457579016685,
 -0.0338699072599411,
 -0.02853497490286827,
 -0.034026388078927994,
 -0.036711424589157104,
 0.06146254390478134,
 -0.006368113681674004,
 -0.008128105662763119,
 0.01568002440035343,
 -0.0

In [17]:
## to embedding documents
text = [doc.page_content for doc in chunks]
document_embeds = embeddings.embed_documents(text[0:100])

In [18]:
print(len(document_embeds))

60


#### Vector Stores

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) takes care of storing embedded data and performing vector search for you.


We use **Chroma** here to demonstrase the vector stores

In [19]:
# 1st initiate chroma object using document chunks and embeddings
doc_search = Chroma.from_documents(chunks, embeddings)

query = "Langchain"

#if we want to find the relevant context for query
docs = doc_search.similarity_search(query)
docs

[Document(id='6eb1e3f7-d40c-481f-a04b-c730a590a786', metadata={'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'page_label': '11', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.26', 'keywords': '', 'trapped': '/False', 'total_pages': 14, 'author': '', 'subject': '', 'moddate': '2024-11-06T10:08:55+00:00', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'page': 10, 'title': '', 'creationdate': '2024-11-06T10:08:55+00:00'}, page_content='leverage LangChain’s extensive library of tools, models, and connectors as part'),
 Document(id='7fe26993-e993-46c8-906b-182641f1825e', metadata={'creationdate': '2024-11-06T10:08:55+00:00', 'keywords': '', 'title': '', 'total_pages': 14, 'page': 0, 'subject': '', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'page_label': '1', 'ptex.fullbanner': 'This i

In [20]:
print(docs[0].page_content)

leverage LangChain’s extensive library of tools, models, and connectors as part


#### Retrievers

A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. Retrievers can be created from vector stores, but are also broad enough to include other sources.
Retrievers accept a string query as input and return a list of Document objects as output.

Note that all vector stores can be cast to retrievers. Refer to the vector store integration docs for available vector stores. This page lists custom retrievers, implemented via subclassing BaseRetriever.

Define **vectore store backbone retriever and parent document retriever**

1. Vectore Store Backbone Retriever

* Note that the results are identical to the ones obtained using the similarity search strategy.


In [21]:
retriever = doc_search.as_retriever()
docs = retriever.invoke("Langchain")
docs[0]

Document(id='6eb1e3f7-d40c-481f-a04b-c730a590a786', metadata={'author': '', 'moddate': '2024-11-06T10:08:55+00:00', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'subject': '', 'total_pages': 14, 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'title': '', 'producer': 'pdfTeX-1.40.26', 'page': 10, 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'keywords': '', 'page_label': '11'}, page_content='leverage LangChain’s extensive library of tools, models, and connectors as part')


#### What is Parent Document Retriever?

The Parent Document Retriever is a retrieval technique used in Retrieval-Augmented Generation (RAG) systems to balance two important requirements:

1. **Accurate retrieval** using small text chunks.
2. **Rich context** using larger documents.

It stores small chunks for vector search while returning larger parent documents during retrieval.

---

#### The Problem

When documents are split for vector databases, there is a trade-off.

#### Small Chunks

**Advantages**
- More precise embeddings
- Better semantic search accuracy

**Disadvantages**
- Limited context
- Important surrounding information may be lost

#### Large Chunks

**Advantages**
- More context for the language model

**Disadvantages**
- Less accurate embeddings
- Retrieval quality may decrease

---

#### How Parent Document Retriever Solves This

Instead of choosing one approach, it combines both.

#### Parent Document
A larger section of text containing complete context.

#### Child Chunks
Smaller pieces created from the parent document.

The vector database stores embeddings for the child chunks, while the original parent document is stored separately.

---

#### Retrieval Process

#### Step 1: Create Parent Documents

A large document is divided into logical sections.

#### Step 2: Split into Child Chunks

Each parent document is broken into smaller chunks.

#### Step 3: Store Embeddings

Embeddings are generated only for the child chunks and stored in the vector database.

#### Step 4: Perform Similarity Search

When a user submits a query, the system searches the vector database and finds the most relevant child chunk.

#### Step 5: Retrieve Parent Document

Instead of returning only the matching child chunk, the system retrieves the corresponding parent document and sends it to the LLM.

---

#### Benefits

#### Accurate Retrieval
Small chunks produce focused embeddings and improve search quality.

#### Better Context
The LLM receives a larger document section instead of an isolated sentence.

#### Improved Answer Quality
More surrounding information helps the model generate complete and accurate answers.

#### Reduced Context Loss
Important details before and after the matching chunk are preserved.

---

#### Typical Configuration

```text
Parent Chunk Size : 1000–2000 characters
Child Chunk Size  : 200–400 characters

In [22]:
parent_splitter = CharacterTextSplitter(chunk_size = 2000, chunk_overlap=20, separator="\n")
child_splitter = CharacterTextSplitter(chunk_size = 400, chunk_overlap=20, separator='\n')

In [23]:
vectore_store = Chroma(
    collection_name="parent-child-collection", embedding_function=embeddings
)
# The storage layer for the parent documents
doc_store = InMemoryStore()

In [24]:
p_retriever = ParentDocumentRetriever(vectorstore=vectore_store,
                                      docstore=doc_store,
                                      parent_splitter=parent_splitter,
                                      child_splitter=child_splitter)

p_retriever.add_documents(document)

In [25]:
docs = p_retriever.invoke("Langchain")

In [27]:
docs[0].page_content

'maintain optimal performance and quickly resolve issues as they arise.\nLangServe’s streamlined workflow ensures that developers can deploy robust\nAPIs with minimal overhead, making it a practical choice for scaling LLM ap-\nplications in production environments.\n4.3 Integration with LangSmith and LangChain\nLangServe integrates seamlessly with LangSmith, which offers observability fea-\ntures like tracing, logging, and performance monitoring. Through LangSmith,\nLangServe users can track metrics such as request frequency, latency, and er-\nror rates. This integration provides a comprehensive view of API performance,\nenabling developers to optimize applications based on real-time data.\nAdditionally, LangServe’s integration with LangChain allows developers to\nleverage LangChain’s extensive library of tools, models, and connectors as part'

#### Memory

Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At bare minimum, a conversational system should be able to access some window of past messages directly.


##### Chat Message History

One of the core utility classes underpinning most (if not all) memory modules is the `ChatMessageHistory` class. This is a super lightweight wrapper that provides convenience methods for saving `HumanMessages`, `AIMessage`s, and then fetching them all.

Here is an example.


In [29]:
history = ChatMessageHistory()

history.add_ai_message("Hi")
history.add_user_message("What is the longest river in the word")

In [30]:
history.messages

[AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the longest river in the word', additional_kwargs={}, response_metadata={})]

In [31]:
ai_response = llm_model.invoke(history.messages)
ai_response

'The Nile River! It stretches for approximately 6,695 kilometers (4,160 miles) from its source in Burundi to its delta on the Mediterranean Sea in Egypt. Would you like to know more about the Nile or another geographical fact?'

#### Conversation Buffer

In [34]:
conversation = ConversationChain(
    llm = llm_model,
    verbose=True,
    memory = ConversationBufferMemory()
)

C:\Users\Vish\AppData\Local\Temp\ipykernel_26096\249880241.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()
C:\Users\Vish\AppData\Local\Temp\ipykernel_26096\249880241.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


In [35]:
conversation.invoke(input="describe honda civic performance")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: describe honda civic performance
AI:

> Finished chain.


{'input': 'describe honda civic performance',
 'history': '',
 'response': "The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enhancing the overall driving experience. This comprehensive system 

In [36]:
conversation.invoke(input="base on your details i want to buy a one. so which year shouldI buy?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: describe honda civic performance
AI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.

The base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.

In terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed

{'input': 'base on your details i want to buy a one. so which year shouldI buy?',
 'history': "Human: describe honda civic performance\nAI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enha

In [37]:
conversation.invoke(input="i am also interested with toyota allion 240. what about it?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: describe honda civic performance
AI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.

The base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.

In terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed

{'input': 'i am also interested with toyota allion 240. what about it?',
 'history': "Human: describe honda civic performance\nAI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enhancing the

ConversationChain(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='describe honda civic performance', additional_kwargs={}, response_metadata={}), AIMessage(content="The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda

### Chains

Chains refer to sequences of calls - whether to an LLM, a tool, or a data preprocessing step.

It combines different LLM calls and actions automatically.

Ex: Summary #1, Summary #2, Summary #3 > Final Summary


In [48]:
template = """Your job is comeup with a classic dish that most popular in given location by the user with a single answer, not descriptively.
                {location}
            your response:
"""

prompt_template = PromptTemplate(template=template, input_variables=['location'])
location_chain = LLMChain(llm=llm_model, prompt=prompt_template, output_key='meal')

location_chain.invoke(input={'location':"china"})

{'location': 'china', 'meal': 'Kung Pao Chicken'}

#### Sequential Chain

In [49]:
template = """Give me the cooking recipe base on the {meal} meal with short and clearly
                your response:
                """

prompt_tem = PromptTemplate(template=template, input_variables=['meal'])
meal_chain = LLMChain(llm=llm_model, prompt=prompt_tem, output_key = 'recipe')


In [50]:
template = """Give me the estimated cooking time base on the recipe given below.
                recipe: {recipe}
                
                your response:
"""

prompt_tem = PromptTemplate(template=template, input_variables=['recipe'])
recipe_chain = LLMChain(llm=llm_model, prompt=prompt_tem, output_key='time')


In [51]:
overall_chain = SequentialChain(chains=[location_chain, meal_chain, recipe_chain],
                                input_variables=['location'],
                                output_variables = ['meal', 'recipe', 'time'],
                                verbose=True)

In [52]:
overall_chain.invoke(input={'location':'Sri Lanka'})



> Entering new SequentialChain chain...

> Finished chain.


{'location': 'Sri Lanka',
 'meal': 'Rice and Curry',
 'recipe': "Here's a simple recipe for a delicious Rice and Curry meal:\n\n**Rice:**\n\n1. Boil 2 cups of water in a pot.\n2. Add 1 cup of uncooked rice, salt to taste.\n3. Cook on medium heat until the water is absorbed.\n\n**Curry:**\n\n1. Heat oil in a pan over medium heat.\n2. Add 1 onion, chopped and sautéed until browned.\n3. Add 2 cloves of garlic, minced and cook for 1 minute.\n4. Add your choice of curry powder or paste (e.g., chicken, beef, or veggie).\n5. Add 1 cup of coconut milk or water.\n6. Simmer for 10-15 minutes until the sauce thickens.\n\n**Serve:**\n\n1. Serve hot rice with the curry on top.",
 'time': "Based on the recipe provided, here are the estimated cooking times:\n\n* **Rice**: approximately 18-20 minutes (boiling water and cooking rice)\n* **Curry**: approximately 15-20 minutes (sauteing onion and garlic, simmering sauce)\n* **Total Cooking Time**: approximately 33-40 minutes\n\nNote: These estimates assu

#### Summarization Chain

In [53]:
web_loader = WebBaseLoader(file_path)
web_data = web_loader.load()

In [58]:
sumarize_chain = load_summarize_chain(llm=llm_model, chain_type="stuff", verbose=False)
sumarize_chain.invoke(input=web_data)

{'input_documents': [Document(metadata={'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf'}, page_content='%PDF-1.5\n%����\n39 0 obj\n<< /Linearized 1 /L 449344 /H [ 2276 299 ] /O 43 /E 141583 /N 14 /T 448841 >>\nendobj\n                                                                                                          \n40 0 obj\n<< /Type /XRef /Length 113 /Filter /FlateDecode /DecodeParms << /Columns 5 /Predictor 12 >> /W [ 1 3 1 ] /Index [ 39 230 ] /Info 37 0 R /Root 41 0 R /Size 269 /Prev 448842                /ID [<673601be49c0f86a8c905684b3678d4b><4de9e18b3876014762f7e01b239139fb>] >>\nstream\nx�cbd`�g`b``8\t"9z��\x06\x10ɤ\rf/\x01�\\1 R�\x10D2�\x05�,� R`\x13�t�\x0b$\x19�dA�\x03 ��\x0eH$o"X�\x1a0�\x1a�w5�\r%��0�{�\x00���q�\x1c%I#�\x17\r�\x1b\x18_\x0f�\x1bF��J\x02\x00�\x0f\x13e\nendstream\nendobj\n                                                                                                                                                         

### Agent

#### Tools

Tools are interfaces that an agent, a chain, or a chat model / LLM can use to interact with the world.

Let’s explore how to work with tools, using the `Python REPL` tool as an example. The `Python REPL` tool can execute Python commands. These commands can either come from the user or be generated by the LLM. This tool is particularly useful for complex calculations. Instead of having the LLM generate the answer directly, it can be more efficient to have the LLM generate code to calculate the answer.


In [2]:
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL

C:\Users\Vish\AppData\Local\Temp\ipykernel_4768\3316118597.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL


In [3]:
python_repl = PythonREPL()

Now, pass simple python command to execute:

In [4]:
python_repl.run("a=3; b=8; print(a+b)")

Python REPL can execute arbitrary code. Use with caution.


'11\n'

ToolKits

Toolkits are collections of tools that are designed to be used together for specific tasks.

Let's create a toolkit that contains one tool which is `PythonREPLTool`. Note that tools are put into a `list` object.


In [5]:
from langchain_experimental.tools import PythonREPLTool

tools = [PythonREPLTool()]

#### Agents

In [6]:
from langchain.agents import create_agent
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate

In [7]:
prompt ="""You are an agent designed to write and execute python code to answer questions.
You have access to a {pythonREPL}, which you can use to execute python code.
If you get an error, debug your code and try again.
Only use the output of your code to answer the question. 
You might know the answer without running any code, but you should still run the code to get the answer.
If it does not seem like you can write code to answer the question, just return "I don't know" as the answer.

"""



In [8]:
agent = create_agent(
    model=llm_model,
    tools=tools,
    system_prompt=prompt
)

NameError: name 'llm_model' is not defined

In [ ]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)

In [ ]:
agent_executor.invoke(input = {"input": "What is the 3rd fibonacci number?"})



> Entering new AgentExecutor chain...


ValueError: contents are required.

### RAG Document Loader (Further...)

In [1]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, JSONLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

C:\Users\Vish\AppData\Local\Temp\ipykernel_13848\1468370060.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader, JSONLoader


In [2]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Q81D33CdRLK6LswuQrANQQ/instructlab.pdf")


In [3]:
data = loader.load()
print(data[0])

page_content='LAB: L ARGE -SCALE ALIGNMENT FOR CHATBOTS
MIT-IBM Watson AI Lab and IBM Research
Shivchander Sudalairaj∗
Abhishek Bhandwaldar∗
Aldo Pareja∗
Kai Xu
David D. Cox
Akash Srivastava∗,†
*Equal Contribution, †Corresponding Author
ABSTRACT
This work introduces LAB (Large-scale Alignment for chatBots), a novel method-
ology designed to overcome the scalability challenges in the instruction-tuning
phase of large language model (LLM) training. Leveraging a taxonomy-guided
synthetic data generation process and a multi-phase tuning framework, LAB sig-
nificantly reduces reliance on expensive human annotations and proprietary mod-
els like GPT-4. We demonstrate that LAB-trained models can achieve compet-
itive performance across several benchmarks compared to models trained with
traditional human-annotated or GPT-4 generated synthetic data. Thus offering a
scalable, cost-effective solution for enhancing LLM capabilities and instruction-
following behaviors without the drawbacks of cata

In [4]:
import wget
import requests

In [5]:
def download(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as file:
            file.write(response.content)

In [6]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/eMSP5vJjj9yOfAacLZRWsg/markdown-sample.md'

In [7]:
download(url, 'markdown_file.md')

In [8]:
loader = UnstructuredMarkdownLoader("markdown_file.md")


In [9]:
markdown_data = loader.load()
markdown_data[0]

Document(metadata={'source': 'markdown_file.md'}, page_content='An h1 header\n\nParagraphs are separated by a blank line.\n\n2nd paragraph. Italic, bold, and monospace. Itemized lists look like:\n\nthis one\n\nthat one\n\nthe other one\n\nNote that --- not considering the asterisk --- the actual text content starts at 4-columns in.\n\nBlock quotes are written like so.\n\nThey can span multiple paragraphs, if you like.\n\nUse 3 dashes for an em-dash. Use 2 dashes for ranges (ex., "it\'s all in chapters 12--14"). Three dots ... will be converted to an ellipsis. Unicode is supported. ☺\n\nAn h2 header\n\nHere\'s a numbered list:\n\nfirst item\n\nsecond item\n\nthird item\n\nNote again how the actual text starts at 4 columns in (4 characters from the left side). Here\'s a code sample:\n\n# Let me re-iterate ...\nfor i in 1 .. 10 { do-something(i) }\n\nAs you probably guessed, indented 4 spaces. By the way, instead of indenting the block, you can use delimited blocks, if you like:\n\ndefine

#### JSON Format

In [10]:
facebook_json_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hAmzVJeOUAMHzmhUHNdAUg/facebook-chat.json"

In [11]:
download(facebook_json_url, "facebook_file.json")

In [12]:
loader = JSONLoader("facebook_file.json", jq_schema=".messages[].content", text_content=True)
json_data = loader.load()

In [13]:
json_data[0:3]

[Document(metadata={'source': 'D:\\Projects\\langchain-explorer\\notebooks\\facebook_file.json', 'seq_num': 1}, page_content='Bye!'),
 Document(metadata={'source': 'D:\\Projects\\langchain-explorer\\notebooks\\facebook_file.json', 'seq_num': 2}, page_content='Oh no worries! Bye'),
 Document(metadata={'source': 'D:\\Projects\\langchain-explorer\\notebooks\\facebook_file.json', 'seq_num': 3}, page_content='No Im sorry it was my mistake, the blue one is not for sale')]

In [14]:
download("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IygVG_j0M87BM4Z0zFsBMA/mlb-teams-2012.csv","mlb_csv.csv")

In [15]:
from langchain_community.document_loaders import CSVLoader

This create multiple document objects each per row

In [16]:
csvloader = CSVLoader(file_path="mlb_csv.csv")
csv_data = csvloader.load()
print(csv_data[0]) 

page_content='Team: Nationals
"Payroll (millions)": 81.34
"Wins": 98' metadata={'source': 'mlb_csv.csv', 'row': 0}


UnstructuredSCVLoader - This create one document object per csv file

In [17]:
from langchain_community.document_loaders import UnstructuredCSVLoader

In [18]:
unstructuredCSVloader = UnstructuredCSVLoader("mlb_csv.csv")
un_csv_data = unstructuredCSVloader.load()
print(un_csv_data[0].page_content)

Team "Payroll (millions)" "Wins" Nationals 81.34 98 Reds 82.20 97 Yankees 197.96 95 Giants 117.62 94 Braves 83.31 94 Athletics 55.37 94 Rangers 120.51 93 Orioles 81.43 93 Rays 64.17 90 Angels 154.49 89 Tigers 132.30 88 Cardinals 110.30 88 Dodgers 95.14 86 White Sox 96.92 85 Brewers 97.65 83 Phillies 174.54 81 Diamondbacks 74.28 81 Pirates 63.43 79 Padres 55.24 76 Mariners 81.97 75 Mets 93.35 74 Blue Jays 75.48 73 Royals 60.91 72 Marlins 118.07 69 Red Sox 173.18 69 Indians 78.43 68 Twins 94.08 66 Rockies 78.06 64 Cubs 88.19 61 Astros 60.65 55


Word File Loads

In [19]:
download("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/94hiHUNLZdb0bLMkrCh79g/file-sample.docx", "sample_word.docx")

In [20]:
from langchain_community.document_loaders import Docx2txtLoader

In [21]:
doc_loader = Docx2txtLoader(file_path="sample_word.docx")
doc_data = doc_loader.load()
print(doc_data[0])

page_content='Demonstration of DOCX support in calibre

This document demonstrates the ability of the calibre DOCX Input plugin to convert the various typographic features in a Microsoft Word (2007 and newer) document. Convert this document to a modern ebook format, such as AZW3 for Kindles or EPUB for other ebook readers, to see it in action.

There is support for images, tables, lists, footnotes, endnotes, links, dropcaps and various types of text and paragraph level formatting.

To see the DOCX conversion in action, simply add this file to calibre using the “Add Books” button and then click “Convert”.  Set the output format in the top right corner of the conversion dialog to EPUB or AZW3 and click “OK”.



Text Formatting

Inline formatting

Here, we demonstrate various types of inline text formatting and the use of embedded fonts.

Here is some bold, italic, bold-italic, underlined and struck out  text. Then, we have a superscript and a subscript. Now we see some red, green and blu

## **Text Splitters**

When using the splitter, you can customize several key parameters to fit your needs:
- **separator**: Define the characters that will be used for splitting the text.
- **chunk_size**: Specify the maximum size of your chunks to ensure they are as granular or broad as needed.
- **chunk_overlap**: Maintain context between chunks by setting the `chunk_overlap` parameter, which determines the number of characters that overlap between consecutive chunks. This helps ensure that information isn't lost at the chunk boundaries.
- **length_function**: Define how the length of chunks is calculated.


In [2]:
import requests

In [3]:
def download (url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as file:
            file.write(response.content)

In [4]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YRYau14UJyh0DdiLDdzFcA/companypolicies.txt"
file_name = "policies.txt"

In [5]:
download(url, file_name)

Load as a document object:

In [1]:
from langchain_community.document_loaders import TextLoader

C:\Users\Vish\AppData\Local\Temp\ipykernel_7804\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
txtloader = TextLoader(file_path="policies.txt")
policy_data = txtloader.load()
policy_data

[Document(metadata={'source': 'policies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continu

### Split By Characters

This is the most basic approach for splitting text, where the content is divided using characters (by default, `"\n\n"`), and chunk length is calculated based on the number of characters.

* **How the text is split:** It is divided at the character level.
* **How chunk size is measured:** By counting characters.

In the code below, you will use `CharacterTextSplitter` to split the document into character-based chunks.

* **Separator:** Set to `''`, meaning every character can act as a split point once the chunk limit is reached.
* **Chunk size:** Set to `200`, so the text is split whenever a chunk exceeds 200 characters.
* **Chunk overlap:** Set to `20`, ensuring that 20 characters are shared between consecutive chunks.
* **Length function:** Uses `len` to measure chunk size.


In [3]:
from langchain_classic.text_splitter import CharacterTextSplitter

In [4]:
txtsplitter = CharacterTextSplitter(
    separator="\n",
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len
)

In [5]:
policytxt = txtsplitter.split_documents(policy_data)
policytxt

[Document(metadata={'source': 'policies.txt'}, page_content="1.\tCode of Conduct\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy."),
 Document(metadata={'source': 'policies.txt'}, page_content="Accountability: We take responsibility for our actions and decisions. We follo

In [6]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [9]:
pipeline = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.2-1B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 512}
)

llm_model = ChatHuggingFace(llm=pipeline)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

### Recursively Split By Character

This text splitter is the recommended one for generic text. It is parameterized by a list of characters, and it tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`.

It processes the large text by attempting to split it by the first character, `\n\n`. If the first split by \n\n results in chunks that are still too large, it moves to the next character, `\n`, and attempts to split by it. This process continues through the list of characters until the chunks are less than the specified chunk size.

This method aims to keep all paragraphs (then sentences, then words) together as much as possible, as these are generally the most semantically related pieces of text.

- **How the text is split**: by list of characters.
- **How the chunk size is measured**: by number of characters.


In [10]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [11]:
recursivesplitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len
)



In [12]:
policy_recursive_splits = recursivesplitter.split_documents(policy_data)
policy_recursive_splits

[Document(metadata={'source': 'policies.txt'}, page_content='1.\tCode of Conduct'),
 Document(metadata={'source': 'policies.txt'}, page_content="Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy."),
 Document(metadata={'source': 'policies.txt'}, page_content="Accountability:

### Code Split

In [13]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter, Language

In [15]:
languages = [e.value for e in Language]
languages

['cpp',
 'go',
 'java',
 'kotlin',
 'js',
 'ts',
 'php',
 'proto',
 'python',
 'r',
 'rst',
 'ruby',
 'rust',
 'scala',
 'swift',
 'markdown',
 'latex',
 'html',
 'sol',
 'csharp',
 'cobol',
 'c',
 'lua',
 'perl',
 'haskell',
 'elixir',
 'powershell',
 'visualbasic6']

In [21]:
PYTHON_CODE = """
    def hello_world():
        print("Hello, World!")
    
    # Call the function
    hello_world()
"""

pythonSplitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size = 200, chunk_overlap =20
) 

code_splits = pythonSplitter.create_documents([PYTHON_CODE])
code_splits

[Document(metadata={}, page_content='def hello_world():\n        print("Hello, World!")\n\n    # Call the function\n    hello_world()')]

### MarkDown Splitter

In [22]:
markdown = md = "# Foo\n\n## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n### Boo \n\nHi this is Lance \n\n## Baz\n\nHi this is Molly"

In [23]:
from langchain_classic.text_splitter import MarkdownHeaderTextSplitter

In [24]:
headers_to_split_on = [
    ("#","Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

markdownSplitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = markdownSplitter.split_text(markdown)

md_header_splits

[Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='Hi this is Jim  \nHi this is Joe'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}, page_content='Hi this is Lance'),
 Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='Hi this is Molly')]

### **Create and Configure a Vector Database to Store Document Embeddings**


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
txtLoader = TextLoader(file_path="policies.txt")
policyData = txtLoader.load()
policyData

[Document(metadata={'source': 'policies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continu

In [8]:
txtSplitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len
)

policy_chunks = txtSplitter.split_documents(policyData)
policy_chunks 

[Document(metadata={'source': 'policies.txt'}, page_content='1.\tCode of Conduct'),
 Document(metadata={'source': 'policies.txt'}, page_content="Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy."),
 Document(metadata={'source': 'policies.txt'}, page_content="Accountability:

### Embedding Model

In [6]:
embeddingModel = HuggingFaceEmbeddings(model="BAAI/bge-m3")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
query = " Hi there how are you?"
query_embeddings = embeddingModel.embed_query(query)
query_embeddings

[-0.023011812940239906,
 0.004767702426761389,
 -0.05313796550035477,
 -0.005050215404480696,
 -0.0016018059104681015,
 -0.08304400742053986,
 -0.019221661612391472,
 -0.0060544004663825035,
 0.022204654291272163,
 0.010639367625117302,
 -0.02716158516705036,
 0.021363142877817154,
 -0.02183602750301361,
 0.008141563273966312,
 -0.002076798817142844,
 -0.019236285239458084,
 -0.019079338759183884,
 -0.052116841077804565,
 -0.01020262110978365,
 -0.04148437827825546,
 -0.03780451789498329,
 0.011413570493459702,
 0.007882311008870602,
 0.0003887656203005463,
 0.036907557398080826,
 0.06060322746634483,
 -0.050083406269550323,
 -0.009613619185984135,
 0.01528336014598608,
 -0.00039318317431025207,
 0.030899103730916977,
 0.01936836540699005,
 0.011768980883061886,
 -0.025885429233312607,
 -0.004284388851374388,
 -0.02982076071202755,
 -0.007756886538118124,
 -0.03250681236386299,
 -0.04153936728835106,
 0.04000617191195488,
 -0.013739874586462975,
 0.014363479800522327,
 0.01903355307877

### Vectore Store

In [11]:
from langchain_classic.vectorstores import Chroma

Next, you need to create an ID list that will be used to assign each chunk a unique identifier, allowing you to track them later in the vector database. The length of this list should match the length of the chunks.

Note: The IDs should be in string format.


In [9]:
ids = [str(i) for i in range(0, len(policy_chunks))]
ids

['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26']

The next step is to use the embedding model to create embeddings for each chunk and then store them in the Chroma database.

The following code demonstrates how to do this.


In [12]:
vectorDb =  Chroma.from_documents(policy_chunks, embeddingModel, ids)

Now that you have built the vector store named `vectordb`, you can use the method `.collection.get()` to print some of the chunks indexed by their IDs.

Note: Although the chunks are stored in the database in embedding format, when you retrieve and print them by their IDs, the database will return the chunk text information instead of the embedding vectors.


In [13]:
for i in range (5):
    print(vectorDb._collection.get(str(i)))

{'ids': ['0'], 'embeddings': None, 'documents': ['1.\tCode of Conduct'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'policies.txt'}]}
{'ids': ['1'], 'embeddings': None, 'documents': ["Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and 

You can also use the method `._collection.count()` to see the length of the vector database, which should be the same as the length of chunks.


In [14]:
vectorDb._collection.count()

27

In [15]:
query = "give me details about email policy"
docs = vectorDb.similarity_search(query)
docs

[Document(metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'policies.txt'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn't interfere with work responsibilities.\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\nConfidentiality: Reserve email for the transmission of confidential information, trade secrets, a

You can specify `k = 1` to just retrieve the top one result.


In [17]:
docs = vectorDb.similarity_search(query, k=2)
docs

[Document(metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'policies.txt'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn't interfere with work responsibilities.\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\nConfidentiality: Reserve email for the transmission of confidential information, trade secrets, a

### FAISS DB

In [18]:
from langchain_classic.vectorstores import FAISS

In [24]:
faissdb = FAISS.from_documents(documents=policy_chunks,embedding=embeddingModel,
                               ids=ids)

In [25]:
ids = [str(i) for i in range(0, len(policy_chunks))]

In [26]:
for i in range ( 5):
    print(faissdb.docstore.search(str(i)))

page_content='1.	Code of Conduct' metadata={'source': 'policies.txt'}
page_content='Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.' metadata={'source': 'policies.txt'}
page_content='Accountability: We take responsibility for our actions and decisions. We follow all relevan

In [27]:
docs = faissdb.similarity_search(query)
docs

[Document(id='6', metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='7', metadata={'source': 'policies.txt'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn't interfere with work responsibilities.\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\nConfidentiality: Reserve email for the transmission of confidential information, 

### Managing vector store: Adding, updating, and deleting entries

There might be situations where new documents come into your RAG application that you want to add to the current vector database, or you might need to delete some existing documents from the database. Additionally, there may be updates to some of the documents in the database that require updating.

The following sections will guide you on how to perform these tasks. You will use the Chroma DB as an example.


#### Add 
Imagine you have a new piece of text information that you want to add to the vector database. First, this information should be formatted into a document object.


In [28]:
from langchain_core.documents import Document

In [29]:
text = "Instructlab is the best open source tool for fine-tuning a LLM."

In [30]:
new_chunk = Document(
    page_content=text,
    metadata = {
        "source": "seran.com",
        "page": 1
    }
)

Then, the new chunk should be put into a list as the vector database only accepts documents in a list.


In [31]:
new_chunk = [new_chunk]

In [32]:
#adding to chromadb
print(vectorDb._collection.get(ids=['215']))


{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [34]:
vectorDb.add_documents(
    documents=new_chunk,
    ids=['215']
)

['215']

In [35]:
print(vectorDb._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['Instructlab is the best open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'seran.com'}]}


In [ ]:
updated_chunk = Document(
    page_content="This is updated document is a perfect open source tool for fine-tuning a LLM.",
    metadata = {
        "source": "seran.com",
        "page": 1
    }
)

In [42]:
vectorDb.update_document(
    document_id='215',
    document=updated_chunk
)



In [43]:
print(vectorDb._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['This is updated document is a perfect open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'seran.com', 'page': 1}]}


In [45]:
vectorDb._collection.delete(ids=['215'])
print(vectorDb._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


# **Develop a Retriever to Fetch Document Segments Based on Queries**


### Overview:

Imagine you are working on a project that involves processing a large collection of text documents, such as research papers, legal documents, or customer service logs. Your task is to develop a system that can quickly retrieve the most relevant segments of text based on a user's query. Traditional keyword-based search methods might not be sufficient, as they often fail to capture the nuanced meanings and contexts within the documents. To address this challenge, you can use different types of retrievers based on LangChain.

Using retrievers is crucial for several reasons:

- Efficiency: Retrievers enable fast and efficient retrieval of relevant information from large datasets, saving time and computational resources.
- Accuracy: By leveraging advanced retrieval techniques, these tools can provide more accurate and contextually relevant results compared to traditional search methods.
- Versatility: Different retrievers can be tailored to specific use cases, making them adaptable to various types of text data and query requirements.
- Context awareness: Some retrievers, like the Parent Document Retriever, can consider the broader context of the document, enhancing the relevance of the retrieved segments.


In [4]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [5]:
def llm_model ():
    
    pipeline_kwargs = {"max_new_tokens":512}
    
    pipeline = HuggingFacePipeline.from_model_id(
        model_id="meta-llama/Llama-3.2-1B-Instruct",
        task="text-generation",
        pipeline_kwargs=pipeline_kwargs
    )
    
    model = ChatHuggingFace(llm=pipeline)
    
    return model

In [6]:
llm_model = llm_model()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Text Splitter

In [7]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [8]:
def text_splitter(data,chunk_size, chunk_overlap):
    txt_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len
    )
    
    splits_documents = txt_splitter.split_documents(data)
    return splits_documents

### Embedding Model

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
embeddings = HuggingFaceEmbeddings(
    model = "BAAI/bge-m3"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Retrivers

A retriever is an interface designed to return documents based on an unstructured query. Unlike a vector store, which stores and retrieves documents, a retriever's primary function is to find and return relevant documents. While vector stores can serve as the backbone of a retriever, there are various other types of retrievers that can be used as well.

Retrievers take a string `query` as input and output a list of `Documents`.



In [12]:
from langchain_classic.document_loaders import TextLoader

In [13]:
policyLoader = TextLoader(file_path="policies.txt")
policy_data = policyLoader.load()
policy_data

[Document(metadata={'source': 'policies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continu

In [15]:
policyChunks = text_splitter(policy_data, 500, 50)
policyChunks

[Document(metadata={'source': 'policies.txt'}, page_content='1.\tCode of Conduct'),
 Document(metadata={'source': 'policies.txt'}, page_content='Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.'),
 Document(metadata={'source': 'policies.txt'}, page_content="Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy."),
 Document(met

### Vectore Store

Store the embeddings into a `ChromaDB`.



In [16]:
from langchain_chroma import Chroma

In [17]:
chroma_db = Chroma.from_documents(policyChunks, embeddings)
chroma_db

In [19]:
chroma_db.similarity_search("Email policy", k=8)

[Document(id='79fbdc25-5eec-4602-bc35-e213b77067dd', metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='14d5ea0c-d09a-4bb6-b89b-9bb487921394', metadata={'source': 'policies.txt'}, page_content='4.\tMobile Phone Policy'),
 Document(id='05f7f9fa-055d-4186-a143-f711b755cace', metadata={'source': 'policies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and follow this policy. Regular reviews ensure its alignment with evolving technology and security standards.'),
 Document(id='1b0e11ad-cd39-40d2-b03c-e75ae2afc8ae', metadata={'source': 'policies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing company matters on public f

Retriver object from chroma db:

In [20]:
retriver = chroma_db.as_retriever()
query = "Email policy"

docs = retriver.invoke(query)
docs

[Document(id='79fbdc25-5eec-4602-bc35-e213b77067dd', metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='14d5ea0c-d09a-4bb6-b89b-9bb487921394', metadata={'source': 'policies.txt'}, page_content='4.\tMobile Phone Policy'),
 Document(id='05f7f9fa-055d-4186-a143-f711b755cace', metadata={'source': 'policies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and follow this policy. Regular reviews ensure its alignment with evolving technology and security standards.'),
 Document(id='1b0e11ad-cd39-40d2-b03c-e75ae2afc8ae', metadata={'source': 'policies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing company matters on public f

In [21]:
retriever = chroma_db.as_retriever(search_type = "mmr")
docs = retriever.invoke(query)
docs

[Document(id='79fbdc25-5eec-4602-bc35-e213b77067dd', metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='1b0e11ad-cd39-40d2-b03c-e75ae2afc8ae', metadata={'source': 'policies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing company matters on public forums or social media.\nHarassment and Inappropriate Content: Internet and email usage must not involve harassment, discrimination, or the distribution of offensive or inappropriate content. Show respect and sensitivity to others in all online communications.'),
 Document(id='5f994e65-6992-4e0d-a207-381c6c86331f', metadata={'source': 'policies.txt'}, page_content='2.\tRecruitment Policy'),
 Document(id='5ab463db-4868-4810-a98e-3f9bd6453510', metadata={'source': 'policies.txt'}, page_content='Policy Review: This policy will undergo pe

#### Similarity score threshold retrieval

You can also set a retrieval method that defines a similarity score threshold, returning only documents with a score above that threshold.


In [22]:
retriever = chroma_db.as_retriever(
    search_type = "similarity_score_threshold", search_kwargs = {"score_threshold":0.4}
)
docs = retriever.invoke(query)
docs

[Document(id='79fbdc25-5eec-4602-bc35-e213b77067dd', metadata={'source': 'policies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='14d5ea0c-d09a-4bb6-b89b-9bb487921394', metadata={'source': 'policies.txt'}, page_content='4.\tMobile Phone Policy'),
 Document(id='05f7f9fa-055d-4186-a143-f711b755cace', metadata={'source': 'policies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and follow this policy. Regular reviews ensure its alignment with evolving technology and security standards.'),
 Document(id='1b0e11ad-cd39-40d2-b03c-e75ae2afc8ae', metadata={'source': 'policies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing company matters on public f

### Multi-Query Retriever




Distance-based vector database retrieval represents queries in high-dimensional space and finds similar embedded documents based on "distance". However, retrieval results may vary with subtle changes in query wording or if the embeddings do not accurately capture the data's semantics.

The `MultiQueryRetriever` addresses this by using an LLM to generate multiple queries from different perspectives for a given user input query. For each query, it retrieves a set of relevant documents and then takes the unique union of these results to form a larger set of potentially relevant documents. By generating multiple perspectives on the same question, the `MultiQueryRetriever` can potentially overcome some limitations of distance-based retrieval, resulting in a richer and more diverse set of results.

Let's consider the query sentence, `"I like cats"`.

On the upper side of the picture, you can see a retriever that relies solely on distance. This retriever calculates the distance between the query and the documents in the vector store, returning the document with the closest match.

On the lower side, you can see a multi-query retriever. It first uses an LLM to generate multiple queries from different perspectives based on the user's input query. For each generated query, it retrieves relevant documents and then returns the union of these results.


In [23]:
import requests


In [24]:
def download(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as file:
            file.write(response.content) 

In [25]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf"
download(url, "langchain.pdf")

In [26]:
from langchain_classic.document_loaders import PyPDFLoader

In [27]:
pdfLoader = PyPDFLoader(file_path="langchain.pdf")
pdf_data = pdfLoader.load()
pdf_data

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'langchain.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental disorders, \nespecially regarding anxiety, depression, and suicidal thoughts, \nunderscores the ne

In [28]:
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'langchain.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that are inherently \ncontext-aware. This means that these applications can \nconnect to a language mod